# Modello CCNL — esecuzione da GitHub
[Apri in Colab](https://colab.research.google.com/github/AntikGhalt/dsge-ccnl-wage-indexation/blob/main/notebooks/run_model_colab.ipynb)

Eseguire le celle in ordine. Non occorre caricare il file Python a mano.
`GIT_REF="main"` scarica il modello corrente; uno SHA fissa la versione.
Una copia già aperta in Colab non aggiorna le proprie celle automaticamente: riaprire
questo link per ricevere gli aggiornamenti del notebook.
Il motore v3 resta sperimentale: [limiti e bug noti](https://github.com/AntikGhalt/dsge-ccnl-wage-indexation/blob/main/docs/validation.md).


In [ ]:
#@title 1. Scarica il modello da GitHub
from pathlib import Path
import subprocess, sys, os, json
REPO_URL = "https://github.com/AntikGhalt/dsge-ccnl-wage-indexation.git"
GIT_REF = "main" # Oppure uno SHA per riprodurre una versione precisa
REPO_DIR = Path("/content/dsge-ccnl-wage-indexation")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
if subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO_DIR, text=True).strip():
    raise RuntimeError("La copia locale contiene modifiche: salvarle prima di aggiornare.")
subprocess.run(["git", "fetch", "origin", GIT_REF], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("Modello caricato dal commit:", COMMIT)


In [ ]:
#@title 2. Esperimenti e parametri personalizzati
from datetime import datetime, timezone
from dataclasses import asdict
from IPython.display import display, Image, Markdown
import importlib.util, shutil
import pandas as pd

EXPERIMENT_MODES = ["institutional"] # Aggiungere "frequency_only" per il controllo
SHOCKS_TO_RUN = ("cost_push", "energy", "technology")
RUN_NO_INDEXATION_BENCHMARK = True
RUN_TERMINAL_CHECK = False
SAVE_TO_DRIVE = False # True per eseguire anche l'ultima cella

# Il JSON nel repository è la fonte dei parametri. Qui inserire solo modifiche.
CUSTOM_PARAMS = {
    # "rho_cost_m": 0.75,
    # "phi_pi": 1.5,
    # "omega_q": 0.65,
    # "wage_drift_speed_m": 0.0,
}
CUSTOM_SHOCK_SIZES = {} # Esempio: {"energy": 0.001}
PARAMS = json.loads((REPO_DIR / "config/baseline.json").read_text())
PARAMS.update(CUSTOM_PARAMS)
MODEL_FILE = REPO_DIR / "model/nk_ccnl_indexation_v3.py"
spec = importlib.util.spec_from_file_location("nkccnl_git", MODEL_FILE)
nk = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = nk
spec.loader.exec_module(nk)
params = nk.Parameters(**PARAMS)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + COMMIT[:8]
OUTPUT_ROOT = REPO_DIR / "outputs" / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
provenance = {"commit": COMMIT, "git_ref": GIT_REF, "parameters": asdict(params),
              "experiment_modes": EXPERIMENT_MODES, "python": sys.version}
(OUTPUT_ROOT / "provenance.json").write_text(json.dumps(provenance, indent=2))
(OUTPUT_ROOT / "environment.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True))
shutil.copy2(MODEL_FILE, OUTPUT_ROOT / MODEL_FILE.name)
display(nk.parameter_guide(params, nk.ExperimentConfig(mode=EXPERIMENT_MODES[0])))
display(nk.equation_guide())
print("ATTENZIONE: v3 ha un bug noto di complementarità/scomposizione; vedere docs/validation.md.")


In [ ]:
#@title 3. Esegui e mostra grafici e tabelle
all_output_folders = []
for mode in EXPERIMENT_MODES:
    settings = json.loads((REPO_DIR / f"config/{mode}.json").read_text())
    settings["shocks_to_run"] = SHOCKS_TO_RUN
    settings["run_no_indexation_benchmark"] = RUN_NO_INDEXATION_BENCHMARK
    settings["shock_sizes"].update(CUSTOM_SHOCK_SIZES)
    cfg = nk.ExperimentConfig(**settings)
    display(nk.specification_table(params, cfg))
    all_output_folders.extend(nk.run_all_default_shocks(params, cfg, base_dir=OUTPUT_ROOT,
                                                       run_terminal_check=RUN_TERMINAL_CHECK))
for folder in all_output_folders:
    display(Markdown(f"## {folder.name}"))
    for _, row in pd.read_csv(folder / "plot_manifest.csv").iterrows():
        display(Image(filename=str(folder / row["file"])))
        display(Markdown(str(row["note"])))
        if isinstance(row.get("warning"), str) and row["warning"].strip():
            display(Markdown(row["warning"]))
    display(pd.read_csv(folder / "irf_summary.csv"))
    benchmark = folder / "benchmark_no_indexation/benchmark_summary.csv"
    if benchmark.exists():
        display(Markdown("### Benchmark senza indicizzazione"))
        display(pd.read_csv(benchmark))
print("Risultati:", OUTPUT_ROOT)


In [ ]:
#@title 4. Salvataggio opzionale su Drive, senza cancellare run precedenti
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DEST = Path("/content/drive/MyDrive/DSGE_CCN_LABOUR/model_outputs_v3") / RUN_ID
    DEST.parent.mkdir(parents=True, exist_ok=True)
    if DEST.exists():
        raise FileExistsError(f"Run già salvata: {DEST}")
    shutil.copytree(OUTPUT_ROOT, DEST)
    print("Run, codice, parametri e commit salvati:", DEST)
else:
    print("Salvataggio Drive disattivato: impostare SAVE_TO_DRIVE = True se desiderato.")
